In [2]:
!pip install transformers datasets sentencepiece sacrebleu sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 12.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [sacrebleu]


In [3]:
from datasets import load_dataset
from transformers import MarianMTModel, MarianTokenizer

dataset = load_dataset(
    "Helsinki-NLP/opus-100",
    "en-ur"
)

print(dataset)
print(dataset["train"][0])

README.md:   0%|          | 0.00/65.4k [00:00<?, ?B/s]

en-ur/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  301kB            

en-ur/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-ur/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148MB            

en-ur/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

en-ur/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  296kB            

en-ur/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/753913 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation'],
        num_rows: 753913
    })
    validation: Dataset({
        features: ['translation'],
        num_rows: 2000
    })
})
{'translation': {'en': 'Yet, remember, as We communed with Moses for forty nights you took the calf in his absence (and worshipped it), and you did wrong.', 'ur': 'اورجب ہم نے موسیٰ سے چالیس رات کا وعدہ کیا پھر اس کے بعد تم نے بچھڑا بنا لیا حالانکہ تم ظالم تھے'}}


In [4]:
model_name = "Helsinki-NLP/opus-mt-en-ur"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

print("Model and Tokenizer Loaded Successfully!")

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/816k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/848k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.39k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  306MB            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Model and Tokenizer Loaded Successfully!


In [5]:
# Display 5 English-Urdu sentence pairs

for i in range(5):
    print("English:", dataset["train"][i]["translation"]["en"])
    print("Urdu:", dataset["train"][i]["translation"]["ur"])
    print("-" * 70)

English: Yet, remember, as We communed with Moses for forty nights you took the calf in his absence (and worshipped it), and you did wrong.
Urdu: اورجب ہم نے موسیٰ سے چالیس رات کا وعدہ کیا پھر اس کے بعد تم نے بچھڑا بنا لیا حالانکہ تم ظالم تھے
----------------------------------------------------------------------
English: Except, among them, Your chosen servants."
Urdu: بجز تیرے اُن بندوں کے جنہیں تو نے خالص کر لیا ہے"
----------------------------------------------------------------------
English: Say, (O Prophet), it was revealed to me that a band of jinn attentively listened to (the recitation of the Qur'an) and then (went back to their people) and said:
Urdu: (اے پیغمبر لوگوں سے) کہہ دو کہ میرے پاس وحی آئی ہے کہ جنوں کی ایک جماعت نے (اس کتاب کو) سنا تو کہنے لگے کہ ہم نے ایک عجیب قرآن سنا
----------------------------------------------------------------------
English: Soon We shall brand him on his snout.
Urdu: ہم عنقریب اس کی ناک پر داغ لگائیں گے
--------------------------------------

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
max_length = 128

def preprocess_function(examples):
    inputs = [item["en"] for item in examples["translation"]]
    targets = [item["ur"] for item in examples["translation"]]

    model_inputs = tokenizer(
        inputs,
        max_length=max_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=targets,
        max_length=max_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_dataset = dataset.map(
    preprocess_function,
    batched=True
)

print(tokenized_dataset)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/753913 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 753913
    })
    validation: Dataset({
        features: ['translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})


In [8]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
tokenized_dataset.save_to_disk(
    "/content/drive/MyDrive/C1_OPUS100_Dataset"
)

print("Dataset saved successfully!")

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

Saving the dataset (0/2 shards):   0%|          | 0/753913 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset saved successfully!


In [1]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

Device: cuda
GPU: Tesla T4


In [10]:
from google.colab import drive
from datasets import load_from_disk

drive.mount('/content/drive')

tokenized_dataset = load_from_disk(
    "/content/drive/MyDrive/C1_OPUS100_Dataset"
)

print(tokenized_dataset)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DatasetDict({
    test: Dataset({
        features: ['translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    train: Dataset({
        features: ['translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 753913
    })
    validation: Dataset({
        features: ['translation', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
})


In [11]:
import torch
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-en-ur"

tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

model = model.to("cuda")

print("Model loaded on:", model.device)

Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

Model loaded on: cuda:0


In [12]:
train_dataset = tokenized_dataset["train"].shuffle(
    seed=42
).select(range(20000))

val_dataset = tokenized_dataset["validation"].select(
    range(1000)
)

print("Training examples:", len(train_dataset))
print("Validation examples:", len(val_dataset))

Training examples: 20000
Validation examples: 1000


In [13]:
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="/content/C1_Translation_Model",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=1,
    weight_decay=0.01,
    fp16=True,
    save_total_limit=1,
    predict_with_generate=False,
    report_to="none"
)

print("Training configuration ready!")

Training configuration ready!


In [15]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer initialized successfully!")

Trainer initialized successfully!


In [16]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.471147,0.876348


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=0.4504974548339844, metrics={'train_runtime': 198.3353, 'train_samples_per_second': 100.839, 'train_steps_per_second': 6.302, 'total_flos': 349905278730240.0, 'train_loss': 0.4504974548339844, 'epoch': 1.0})

In [17]:
test_dataset = tokenized_dataset["test"]
print("Test examples:", len(test_dataset))

Test examples: 2000


In [19]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared!")

GPU memory cleared!


In [21]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()

test_subset = test_dataset.select(range(500))

all_predictions = []
all_references = []

model.eval()

for i in range(0, 500, 4):
    batch = test_subset[i:i+4]

    inputs = tokenizer.pad(
        {
            "input_ids": batch["input_ids"],
            "attention_mask": batch["attention_mask"]
        },
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_length=128,
            num_beams=2
        )

    decoded = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True
    )

    references = tokenizer.batch_decode(
        batch["labels"],
        skip_special_tokens=True
    )

    all_predictions.extend(decoded)
    all_references.extend(references)

    if (i + 4) % 100 == 0:
        print("Processed:", min(i + 4, 500), "/ 500")

print("Total predictions:", len(all_predictions))
print("Total references:", len(all_references))

Processed: 100 / 500
Processed: 200 / 500
Processed: 300 / 500
Processed: 400 / 500
Processed: 500 / 500
Total predictions: 500
Total references: 500


In [22]:
import sacrebleu

bleu = sacrebleu.corpus_bleu(
    all_predictions,
    [all_references]
)

chrf = sacrebleu.corpus_chrf(
    all_predictions,
    [all_references]
)

print("BLEU Score:", round(bleu.score, 2))
print("chrF Score:", round(chrf.score, 2))

BLEU Score: 23.02
chrF Score: 43.55


In [23]:
for i in range(5):
    print("English:", test_dataset[i]["translation"]["en"])
    print("Actual Urdu:", test_dataset[i]["translation"]["ur"])
    print("Predicted Urdu:", all_predictions[i])
    print("-" * 60)

English: My 50 hours of community service meant cleaning a city I didn't care about.
Actual Urdu: کے بارے میں میں پرواہ نہیں ایک شہر کی صفائی کا مطلب کمیونٹی کی خدمت میری 50 گھنٹے
Predicted Urdu: مقامی سروس کے 50 گھنٹے کا مطلب تھا کہ میں نے اس شہر کو صاف نہیں کیا تھا.
------------------------------------------------------------
English: UM, HE'S MY MOM'S BOYFRIEND.
Actual Urdu: UM، وہ میری ماں اور پریمی کے.
Predicted Urdu: حضرات، وہ میری ماں کے دوست ہے.
------------------------------------------------------------
English: Those [believers] who responded to Allah and the Messenger after injury had struck them. For those who did good among them and feared Allah is a great reward -
Actual Urdu: جن لوگوں نے زخم کھانے کے بعد بھی اللہ اور رسول کی پکار پر لبیک کہا اُن میں جو اشخاص نیکوکار اور پرہیز گار ہیں اُن کے لیے بڑا اجر ہے
Predicted Urdu: جن لوگوں نے زخم کھا چکنے کے بعد بھی اللہ اور رسول (صلی اللہ علیہ وآلہ وسلم) کے حکم پر لبیک کہا، اُن میں جو صاحبانِ اِحسان ہیں اور پرہیزگار ہیں، ان کے ل

In [24]:
save_path = "/content/drive/MyDrive/C1_Translation_Model"

trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print("Model and tokenizer saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer saved successfully!
